# RewardBench — the 0.8025 1.5B reward model (a discriminating yardstick)

Eval-only (~20-30 min on a T4): score the **best RM (0.8025, Qwen2.5-1.5B)** on RewardBench and report
the per-category breakdown (Chat / Chat Hard / Safety / Reasoning) + category-mean. RewardBench is
**non-saturated** — unlike HH-RLHF where every UF-trained RM sits at chance — so it can finally tell
0.726 / 0.8025 / mixed RMs apart, *before* spending 12 h on a 1.5B-RM → PPO → judge chain.

**Prerequisite:** `+ Add Input` the dataset `georgezhang06/rlhf-rm-1p5b-08025`.

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print(f'GPU: {name} cap={cap} -> '+('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python -m pytest tests/ -q 2>&1 | tail -2   # sanity: RewardBench aggregation test included

## Locate the attached 0.8025 reward model

In [ ]:
import glob, os, shutil, json
P100=json.load(open('/tmp/gpu.json'))['p100']
DTYPE=('float32') if P100 else ('bfloat16')
hits=glob.glob('/kaggle/input/**/reward_config.json', recursive=True)
assert hits, 'No reward model under /kaggle/input. + Add Input -> georgezhang06/rlhf-rm-1p5b-08025.'
def _wsz(d):
    w=glob.glob(d+'/*.safetensors')+glob.glob(d+'/*.bin')
    return max([os.path.getsize(f) for f in w], default=0)
cands=sorted((os.path.dirname(h) for h in hits if '/smoke/' not in h), key=_wsz, reverse=True)
assert cands, 'only smoke checkpoints found'
RM_SRC=cands[0]; os.makedirs('checkpoints', exist_ok=True)
if os.path.abspath(RM_SRC)!=os.path.abspath('checkpoints/reward_model'):
    shutil.rmtree('checkpoints/reward_model', ignore_errors=True); shutil.copytree(RM_SRC,'checkpoints/reward_model')
sz=_wsz('checkpoints/reward_model'); print(f'RM <- {RM_SRC} ({sz/1e6:.0f} MB) dtype={DTYPE}')
assert sz>1e8, 'RM weights too small/partial for a 1.5B model.'

## Run RewardBench

In [ ]:
import subprocess
cmd=('python scripts/eval_rewardbench.py --reward-model checkpoints/reward_model '
     '--device cuda --batch-size 16 --max-length 1024')
print('$',cmd,flush=True)
o=subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(o.stdout[-1500:])
if o.returncode: print('STDERR:', o.stderr[-2000:])
allout=o.stdout+'\n'+o.stderr
rep=[l for l in allout.splitlines() if 'rewardbench report' in l]
md=('# RewardBench — Qwen2.5-1.5B 0.8025 RM\n\n## Scoreboard\n```\n'+o.stdout[-1200:]+'\n```\n'
    +('\n## Full per-subset\n```\n'+rep[-1][-1600:]+'\n```\n' if rep else '')
    +('\n## stderr (error)\n```\n'+o.stderr[-1500:]+'\n```\n' if o.returncode else ''))
open('/kaggle/working/RESULTS.md','w').write(md); print('\nsaved -> RESULTS.md')

### Read
Compare the category-mean to the v15 0.5B RM (run locally) — if the 1.5B clearly wins on
**Chat Hard / Reasoning** (the discriminating categories), RewardBench confirms RM quality scales,
and it's the right cheap yardstick to pick which RM to invest a full PPO+judge run on.